# CS2 EXP-3 — NeoBERT-250M Frozen Encoder Linear Probe

## 1. Dependencies

In [1]:
import sys
import subprocess

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "--extra-index-url", "https://download.pytorch.org/whl/cu121",
    "torch==2.5.1",
    "torchvision",
    "torchaudio",
    "transformers<4.49.0",  # Avoids the CVE check enforcing PyTorch 2.6
    "numpy<2.1.0",
    "pandas",
    "scikit-learn",
    "matplotlib",
    "pyarrow",
    "joblib",
    "tqdm",
    "psutil",
    "einops",   # NeoBERT (chandar-lab/NeoBERT) remote modeling code dependency
], check=True)

# xformers is pinned to match torch==2.5.1 exactly (unpinned installs pull a
# newer release that silently upgrades torch out from under this pin).
# flash-attention is not required since `use_unpadding=False` in models.py.
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "--no-deps",
    "xformers==0.0.28.post3",
], check=True)

import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
print("Dependencies installed successfully for CUDA 12.1 driver!")


Dependencies installed successfully for CUDA 12.1 driver!


In [ ]:
import subprocess
nvidia_smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(nvidia_smi.stdout or "nvidia-smi produced no stdout")
if nvidia_smi.stderr:
    print(nvidia_smi.stderr)

import os
print("CUDA_VISIBLE_DEVICES before override:", repr(os.environ.get("CUDA_VISIBLE_DEVICES")))

import torch
print("torch.__version__:", torch.__version__)
print("torch.version.cuda:", torch.version.cuda)
print("torch.cuda.is_available():", torch.cuda.is_available())
print("torch.cuda.device_count():", torch.cuda.device_count())


In [3]:
import torch
try:
    torch.cuda.init()
except Exception as e:
    print(repr(e))


In [4]:
import subprocess, sys

# Fallback: only needed if the cell above shows torch.cuda.is_available()==False
# after the pinned install (seen occasionally on some CUDA 12.1 hosts). Re-run
# this, then RESTART THE KERNEL, then re-run from the top -- do not just
# continue in the same process, torch's CUDA init is one-shot per process.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torch", "torchvision", "torchaudio"], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "torch", "torchvision", "torchaudio",
    "--index-url", "https://download.pytorch.org/whl/cu121",
], check=True)

print("Reinstalled. Restart the kernel now, then re-run your CUDA check cell.")


Found existing installation: torch 2.5.1+cu121
Uninstalling torch-2.5.1+cu121:
  Successfully uninstalled torch-2.5.1+cu121
Found existing installation: torchvision 0.20.1+cu121
Uninstalling torchvision-0.20.1+cu121:
  Successfully uninstalled torchvision-0.20.1+cu121
Found existing installation: torchaudio 2.11.0
Uninstalling torchaudio-2.11.0:
  Successfully uninstalled torchaudio-2.11.0


Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached https://download-r2.pytorch.org/whl/cu121/torch-2.5.1%2Bcu121-cp310-cp310-linux_x86_64.whl (780.4 MB)
  Using cached https://download-r2.pytorch.org/whl/cu121/torchvision-0.20.1%2Bcu121-cp310-cp310-linux_x86_64.whl (7.3 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 35.5 MB/s eta 0:00:00


Reinstalled. Restart the kernel now, then re-run your CUDA check cell.


In [5]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch

assert torch.cuda.is_available(), "CUDA is not available!"
assert torch.cuda.device_count() == 1, f"Expected 1 GPU, but PyTorch sees {torch.cuda.device_count()}"

DEVICE = "cuda:0"

print(f"Locked to single GPU: {torch.cuda.get_device_name(0)}")
print(f"Total Visible GPUs in PyTorch: {torch.cuda.device_count()}")


Locked to single GPU: NVIDIA A100-SXM4-80GB
Total Visible GPUs in PyTorch: 1


## 1.5 Settings

In [ ]:
from pathlib import Path
import os

REPO_URL = "https://github.com/EnomisLP/DiverseVul--IS-Project.git"
REPO_BRANCH = "Recovery"

WORKSPACE_ROOT = Path.cwd()
REPO_ROOT = WORKSPACE_ROOT / "DiverseVul--IS-Project"
PROJECT_DIR = REPO_ROOT / "vuln-detection"
SRC_DIR = PROJECT_DIR / "src"

DATA_ROOT = WORKSPACE_ROOT / "IntelligentSystemProject" / "VulnerabilityDetectionData"
PROCESSED_DIR = DATA_ROOT / "processed"
MANIFEST_ROOT = DATA_ROOT / "manifests"
OUTPUT_ROOT = DATA_ROOT / "outputs"

DOWNSAMPLED_PARQUET = PROCESSED_DIR / "rdiversevul_cs1_normalized_plus_abstracted_v2_downsampled20k.parquet"
MANIFEST_PATH = MANIFEST_ROOT / "cs1_shared_rotating_5fold_v1" / "project_grouped_5fold_manifest.parquet"

CODE_COLUMN = "normalized_code"
# CODE_COLUMN = "abstracted_code_v1"
CODE_COLUMN_TAG = "abstracted" if CODE_COLUMN == "abstracted_code_v1" else "normalized"

EXP3_OUTPUT_DIR = OUTPUT_ROOT / "case_study_2" / f"exp3_neobert_linear_probe_v1_{CODE_COLUMN_TAG}"
EXP3_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDING_CACHE_DIR = DATA_ROOT / "embedding_cache" / "neobert_v1" / CODE_COLUMN_TAG
EMBEDDING_CACHE_DIR.mkdir(parents=True, exist_ok=True)

HF_CACHE_DIR = WORKSPACE_ROOT / "IntelligentSystemProject" / "hf_cache"
# os.environ["HF_TOKEN"] = "secret"

RUN_SMOKE_TEST = True
RUN_PROFILE = True
RUN_OFFICIAL = True

print("Settings loaded.")
print(f"Workspace: {WORKSPACE_ROOT}")
print(f"Repository: {REPO_ROOT}")
print(f"Data root: {DATA_ROOT}")
print(f"Downsampled parquet: {DOWNSAMPLED_PARQUET}")
print(f"Manifest: {MANIFEST_PATH}")
print(f"Hugging Face cache: {HF_CACHE_DIR}")


## 2. Clone the repository

In [ ]:
import urllib.request
import zipfile
from pathlib import Path

if not REPO_ROOT.exists():
    print(f"Downloading repository (branch: {REPO_BRANCH}) without git...")

    clean_url = REPO_URL.removesuffix(".git")
    zip_url = f"{clean_url}/archive/refs/heads/{REPO_BRANCH}.zip"
    zip_path = Path.cwd() / "repo_temp.zip"

    urllib.request.urlretrieve(zip_url, zip_path)

    print("Extracting files...")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(Path.cwd())

    repo_name = clean_url.split("/")[-1]
    extracted_folder = Path.cwd() / f"{repo_name}-{REPO_BRANCH}"
    if extracted_folder.exists():
        extracted_folder.rename(REPO_ROOT)

    zip_path.unlink()
    print("Repository setup complete!")
else:
    print(f"Repository already exists at {REPO_ROOT}")


## 3. Verify GPU, RAM, and storage budget

In [8]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import torch
assert torch.cuda.is_available(), "CUDA is not available!"
assert torch.cuda.device_count() == 1, f"Expected 1 GPU, but PyTorch sees {torch.cuda.device_count()}"
DEVICE = "cuda:0"
print(f"Locked to single GPU: {torch.cuda.get_device_name(0)}")
print(f"bfloat16 supported: {torch.cuda.is_bf16_supported()}")
print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


Locked to single GPU: NVIDIA A100-SXM4-80GB
bfloat16 supported: True
Total VRAM: 84.99 GB


## 4. Data availability check

In [ ]:
STORAGE_CAP_GB = 60

required_data_files = {
    "downsampled parquet": DOWNSAMPLED_PARQUET,
    "shared 5-fold manifest": MANIFEST_PATH,
}

missing = {name: path for name, path in required_data_files.items() if not path.is_file()}

if missing:
    print("Missing required data files:")
    for name, path in missing.items():
        print(f"  - {name}: {path}")
    print(
        "\nThese files are produced by notebooks/scope2_preprocessing.ipynb (downsampling + "
        "manifest-generation sections) and were previously synced through Google Drive. Copy "
        f"them into the paths above, or re-run that notebook. Keep an eye on the {STORAGE_CAP_GB} GB storage cap."
    )
    raise FileNotFoundError("Required processed data/manifest are missing; see instructions above.")
else:
    for name, path in required_data_files.items():
        size_mb = path.stat().st_size / 1e6
        print(f"Found {name}: {path} ({size_mb:.1f} MB)")


## 5. Verify required repository files

In [ ]:
required_repo_files = [
    SRC_DIR / "case_study_2" / "models.py",
    SRC_DIR / "case_study_2" / "data_loader.py",
    SRC_DIR / "case_study_2" / "exp3" / "exp3_linear_probe.py",
]

missing_repo_files = [str(path) for path in required_repo_files if not path.exists()]
if missing_repo_files:
    raise FileNotFoundError("Required EXP-3 files are missing:\n" + "\n".join(missing_repo_files))

print("Required Case Study 2 files are present.")

## 6. Import project modules

In [ ]:
import sys

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

for mod_name in list(sys.modules.keys()):
    if mod_name.startswith("case_study_2") or mod_name.startswith("case_study_1") or mod_name.startswith("utils"):
        del sys.modules[mod_name]

from case_study_2.data_loader import create_dataloader
from case_study_2.models import (
    configure_huggingface_cache, load_code_tokenizer, load_code_encoder,
    DEFAULT_NEOBERT_MODEL, DEFAULT_NEOBERT_TOKENIZER,
)
from case_study_2.exp3.exp3_linear_probe import (
    Exp3Config, NestedProbeConfig, extract_embeddings,
    run_exp3_nested_inner_profile, run_exp3_nested_probe,
)
from utils import split_manifest
from utils import evaluation
from utils.confidence_intervals import bootstrap_metric_ci, format_ci_report

print("Imported. Model:", DEFAULT_NEOBERT_MODEL)


## 7. Load the downsampled dataset and shared 5-fold manifest

In [ ]:
import pandas as pd

if not DOWNSAMPLED_PARQUET.is_file():
    raise FileNotFoundError(f"Missing downsampled parquet: {DOWNSAMPLED_PARQUET}")
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"Missing manifest: {MANIFEST_PATH}")

full_df = pd.read_parquet(DOWNSAMPLED_PARQUET)
manifest_df = split_manifest.load_manifest(MANIFEST_PATH, config=split_manifest.SplitConfig(n_splits=5, random_state=42))

print("full_df rows:", len(full_df))
print("manifest_df rows:", len(manifest_df))

fold_summary_diag = split_manifest.summarize_manifest(
    manifest_df, config=split_manifest.SplitConfig(n_splits=5, random_state=42)
)
display(fold_summary_diag)

fold_size_ratio = fold_summary_diag["test_rows"].max() / fold_summary_diag["test_rows"].min()
print(f"Fold test-size balance: smallest={fold_summary_diag['test_rows'].min()} rows, "
      f"largest={fold_summary_diag['test_rows'].max()} rows, ratio={fold_size_ratio:.2f}x")
if fold_size_ratio > 2.0:
    print("WARNING: fold sizes are notably imbalanced (ratio > 2x) -- "
          "regenerate the manifest via scope2_preprocessing.ipynb with an updated "
          "DOWNSAMPLE_MAX_ROWS_PER_PROJECT.")
else:
    print("Fold sizes look reasonably balanced.")


## 8. Build the dataset frame

In [ ]:
required_columns = {"source_row_id", "normalized_code", "abstracted_code_v1", "label", "project"}
missing_columns = required_columns - set(full_df.columns)
if missing_columns:
    raise ValueError(f"Missing columns in full_df: {missing_columns}")

full_indexed = full_df.set_index("source_row_id", drop=False)
manifest_ids = set(manifest_df["source_row_id"].tolist())
if manifest_ids != set(full_indexed.index):
    raise RuntimeError(
        "Manifest coverage does not match the downsampled dataset exactly. "
        f"Missing={len(set(full_indexed.index) - manifest_ids)}, extra={len(manifest_ids - set(full_indexed.index))}"
    )

dataset_frame = full_indexed.loc[list(manifest_ids)].reset_index(drop=True)
print("dataset_frame rows:", len(dataset_frame))
print("Unique projects:", dataset_frame["project"].nunique())


## 9. Configuration objects

In [ ]:
base_config = Exp3Config(hf_cache_dir=HF_CACHE_DIR, code_column=CODE_COLUMN)
nested_config = NestedProbeConfig()

print("Code column:", base_config.code_column)
print("C_grid:", nested_config.C_grid)
print("inner_n_splits:", nested_config.inner_n_splits)


## 10. Smoke test on a small subsample

NeoBERT ships as `trust_remote_code` and has a known numerical-instability bug (PDD sec. 5.2 / GitHub Issue #11). Cheap sanity check on ~300 rows -- tokenizer, encoder loading, embedding extraction (with the NaN guardrail), and a quick probe fit -- before committing to the full development+holdout embedding extraction.

In [ ]:
if RUN_SMOKE_TEST:
    smoke_df = dataset_frame.sample(n=min(300, len(dataset_frame)), random_state=42).reset_index(drop=True)

    print("Loading tokenizer...")
    _smoke_tokenizer = load_code_tokenizer(base_config.tokenizer_name, hf_cache_dir=HF_CACHE_DIR)
    print("Loading encoder...")
    _smoke_encoder = load_code_encoder(
        base_config.model_name, dtype_policy=base_config.dtype_policy,
        device=DEVICE, freeze=True, hf_cache_dir=HF_CACHE_DIR,
    )

    _smoke_embeddings = extract_embeddings(
        _smoke_encoder, _smoke_tokenizer, smoke_df, base_config, DEVICE,
        cache_path=None,
    )
    print("Smoke embeddings shape:", _smoke_embeddings.shape)
    assert _smoke_embeddings.shape[0] == len(smoke_df)
    import numpy as np
    assert np.isfinite(_smoke_embeddings).all(), "Non-finite values slipped past the guardrail -- investigate before proceeding."

    print("Smoke test passed: encoder loads, tokenizes, extracts finite embeddings.")

    del _smoke_encoder, _smoke_tokenizer, _smoke_embeddings
    torch.cuda.empty_cache()
else:
    print("RUN_SMOKE_TEST=False; skipping.")


## 11. Extract frozen NeoBERT embeddings for the full dataset

In [ ]:
try:
    print("Loading tokenizer...")
    tokenizer = load_code_tokenizer(base_config.tokenizer_name, hf_cache_dir=HF_CACHE_DIR)
    print("Tokenizer loaded")

    print("Loading encoder...")
    encoder = load_code_encoder(
        base_config.model_name,
        dtype_policy=base_config.dtype_policy,
        device=DEVICE,
        freeze=True,
        hf_cache_dir=HF_CACHE_DIR,
    )
    print("Encoder loaded successfully on", DEVICE)

    print("\nExtracting embeddings for the full dataset...")
    dataset_embeddings = extract_embeddings(
        encoder, tokenizer, dataset_frame, base_config, DEVICE,
        cache_path=EMBEDDING_CACHE_DIR / "dataset_embeddings.npy",
    )
    print("Embeddings extracted:", dataset_embeddings.shape)

    del encoder
    torch.cuda.empty_cache()
    print("VRAM cleaned up")

except RuntimeError as e:
    if "NaN/Inf detected" in str(e):
        print("ERROR: NeoBERT numerical-instability bug triggered")
        print(str(e))
        print("Do not silently drop/zero these rows. Investigate dtype_policy "
              "(try dtype_policy='float32' on base_config) before re-running.")
    raise
except Exception as e:
    print(f"ERROR during embedding extraction: {type(e).__name__}: {e}")
    import traceback
    traceback.print_exc()
    raise


## 12. Single-fold inner profile

In [ ]:
if RUN_PROFILE:
    profile = run_exp3_nested_inner_profile(
        dataset_frame=dataset_frame,
        dataset_embeddings=dataset_embeddings,
        manifest=manifest_df,
        outer_fold_id=4,
        base_config=base_config,
        nested_config=nested_config,
    )
    print("Profile duration minutes:", profile["total_profile_seconds"] / 60)
    display(pd.DataFrame([profile["selected"]]))
    display(profile["C_summary"])
else:
    print("RUN_PROFILE=False; skipping.")


## 13. Official rotating 5-fold run

In [ ]:
if RUN_OFFICIAL:
    results = run_exp3_nested_probe(
        dataset_frame=dataset_frame,
        dataset_embeddings=dataset_embeddings,
        manifest=manifest_df,
        base_config=base_config,
        nested_config=nested_config,
        output_dir=EXP3_OUTPUT_DIR,
        additional_metadata={
            "input_parquet": str(DOWNSAMPLED_PARQUET),
            "manifest_path": str(MANIFEST_PATH),
        },
    )
    print("Pooled OOF metrics (secondary cross-check):")
    display(pd.DataFrame([{"metric": k, "value": v} for k, v in results["evaluation"]["pooled_metrics"].items()]))
    print("Mean +/- std across the 5 outer folds (headline result):")
    display(results["evaluation"]["fold_summary"])
    print("Selected hyperparameters by outer fold:")
    display(results["selected"])
else:
    results = None
    print("RUN_OFFICIAL=False; skipping.")


## 14. Confidence interval on pooled OOF PR-AUC (ad hoc)

In [ ]:
exp3_oof_ci = bootstrap_metric_ci(
    results["oof_predictions"],
    metric="average_precision_pr_auc",
    n_bootstrap=1000,
    random_state=42,
)
print(format_ci_report(exp3_oof_ci))


## 15. Cleanup

In [ ]:
import gc
import shutil
import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("VRAM allocated:", torch.cuda.memory_allocated() / 1e9, "GB")

total, used, free = shutil.disk_usage(WORKSPACE_ROOT)
print(f"Disk usage at {WORKSPACE_ROOT}: {used/1e9:.1f} GB used / {total/1e9:.1f} GB total ({free/1e9:.1f} GB free)")
if used / 1e9 > STORAGE_CAP_GB:
    print(f"WARNING: workspace usage exceeds the {STORAGE_CAP_GB} GB storage cap -- consider pruning old checkpoints under {EXP3_OUTPUT_DIR}.")
